In [ ]:
import pandas as pd
import re

In [20]:
df = pd.read_json(r'D:\IMP  ML  PROJECTS\CAR PRICE PREDICTION\web scraping\extraction\car_dataset_ahmedabad.json')
df.head()

,url,car_name,Price,Registration Year,Insurance,Fuel Type,Seats,Kms Driven,RTO,Ownership,...,Secondary Fuel Type,Drag Coefficient,Boot Space Rear Seat Folding,Approach Angle,Break-over Angle,Departure Angle,Petrol Mileage (ARAI),Petrol Fuel Tank Capacity (Litres),Acceleration 0-100kmph,CNG Highway Mileage
0,https://www.cardekho.com/used-car-details/used...,Hyundai Grand i10,₹3 Lakh,2015,Comprehensive,Petrol,5 Seats,"1,10,422 Kms",Ahmedabad,Second Owner,...,None,None,None,None,None,None,NaN,NaN,NaN,None
1,https://www.cardekho.com/buy-used-car-details/...,Hyundai i10,₹3.22 Lakh,Apr 2015,Comprehensive,Petrol,5 Seats,"41,223 Kms",Ahmedabad,First Owner,...,None,None,None,None,None,None,NaN,NaN,NaN,None
2,https://www.cardekho.com/buy-used-car-details/...,Ford Figo,₹4.13 Lakh,Dec 2019,Comprehensive,Petrol,5 Seats,"73,627 Kms",Gandhinagar,First Owner,...,None,None,None,None,None,None,NaN,NaN,NaN,None
3,https://www.cardekho.com/buy-used-car-details/...,Hyundai Grand i10,₹4.37 Lakh,Jul 2017,Comprehensive,Petrol,5 Seats,"72,080 Kms",Surat,First Owner,...,None,None,None,None,None,None,NaN,NaN,NaN,None
4,https://www.cardekho.com/used-car-details/used...,Chevrolet Beat,₹2.51 Lakh,2015,-,Diesel,5 Seats,"1,00,000 Kms",Rajkot,Third Owner,...,None,None,None,None,None,None,NaN,NaN,NaN,None


In [21]:
req_col = []
    
with open(r'D:\IMP  ML  PROJECTS\CAR PRICE PREDICTION\price\features.txt', 'r') as f:
    features = f.read().split('\n')

for i in features:
    if i in df.columns:
        req_col.append(i)
        
df = df[req_col]

# Change the df to 'dataset.csv'
df.to_csv('dataset.csv')

In [22]:
df.head()

,car_name,Price,Registration Year,Kms Driven,Ownership,Fuel,Transmission,Drive Type,Engine,Power,Mileage,No. of Cylinders,Turbo Charger,Seats,Kerb Weight,Ground Clearance Unladen,Petrol Fuel Tank Capacity,Diesel Fuel Tank Capacity,CNG Fuel Tank Capacity
0,Hyundai Grand i10,₹3 Lakh,2015,"1,10,422 Kms",Second Owner,Petrol,Manual,FWD,1197 cc,82 bhp,18.9 kmpl,4.0,No,5 Seats,935 kg,165 mm,43 Litres,None,None
1,Hyundai i10,₹3.22 Lakh,Apr 2015,"41,223 Kms",First Owner,Petrol,Manual,FWD,1086 cc,68.05 bhp,19.81 kmpl,4.0,No,5 Seats,860 kg,165 mm,35 Litres,None,None
2,Ford Figo,₹4.13 Lakh,Dec 2019,"73,627 Kms",First Owner,Petrol,Manual,Front Wheel Drive,1196 cc,70 bhp,15.6 kmpl,4.0,No,5 Seats,1090 kg,168 mm,45 Litres,None,None
3,Hyundai Grand i10,₹4.37 Lakh,Jul 2017,"72,080 Kms",First Owner,Petrol,Manual,FWD,1197 cc,81.86 bhp,18.9 kmpl,4.0,No,5 Seats,1060 kg,165 mm,43 Litres,None,None
4,Chevrolet Beat,₹2.51 Lakh,2015,"1,00,000 Kms",Third Owner,Diesel,Manual,FWD,936 cc,56.3 bhp,25.44 kmpl,3.0,Yes,5 Seats,1025 kg,175 mm,None,35 Litres,None


In [23]:
# Drop duplicates 
df.drop_duplicates(inplace=True)

In [24]:
# Percentage missing in each feature
for col in df.columns:
    percent_missing = df[col].isnull().sum() / len(df)
    print(f'{col}: {percent_missing:.2%}')

car_name: 0.00%
Price: 0.19%
Registration Year: 0.28%
Kms Driven: 0.19%
Ownership: 0.28%
Fuel: 11.52%
Transmission: 0.19%
Drive Type: 13.67%
Engine: 1.22%
Power: 2.90%
Mileage: 4.87%
No. of Cylinders: 0.75%
Turbo Charger: 16.10%
Seats: 0.75%
Kerb Weight: 7.87%
Ground Clearance Unladen: 30.90%
Petrol Fuel Tank Capacity: 27.25%
Diesel Fuel Tank Capacity: 79.12%
CNG Fuel Tank Capacity: 97.38%


### Target Encoding Function

In [25]:
def weighted_target_encoding(df, feature, target, m=10):
    
    # global mean of target
    global_mean = df[target].mean()
    
    # compute category stats
    stats = df.groupby(feature)[target].agg(['mean', 'count'])
    
    # weighted mean formula
    stats['weighted_mean'] = (
        stats['mean'] * stats['count'] + global_mean * m
    ) / (stats['count'] + m)
    
    # mapping dictionary
    mapping = stats['weighted_mean'].to_dict()
    
    # apply encoding
    encoded_feature = df[feature].map(mapping)
    
    return encoded_feature, mapping

0. Price

In [26]:
def clean_price(x):
    x = str(x).replace("₹", "").strip()

    if "Lakh" in x:
        return float(x.replace("Lakh", "").strip()) * 100000

    elif "Crore" in x:
        return float(x.replace("Crore", "").strip()) * 10000000

    elif "Thousand" in x:
        return float(x.replace("Thousand", "").strip()) * 1000

    else:
        return None


df["Price"] = df["Price"].apply(clean_price)

In [27]:
df['Price'] = df['Price'].fillna(df['Price'].median())

1. 'car_name'

In [28]:
import re

def clean_car_name(df):

    # remove extra spaces
    df['car_name'] = df['car_name'].str.strip()

    # remove rows that contain scraping phrases
    pattern = r"used cars for sale|in Ahmedabad|₹"

    df = df[~df['car_name'].str.contains(pattern, case=False, na=False)]

    # keep only names that start with letters
    df = df[df['car_name'].str.match(r'^[A-Za-z]', na=False)]

    return df

In [29]:
df = clean_car_name(df)

df[['brand', 'model']] = df['car_name'].str.split(' ', n=1, expand=True)

In [30]:
df['model'].nunique()

145

In [31]:
df['model'], model_mapping = weighted_target_encoding(
    df,
    feature="model",
    target="Price",
    m=20
)

df['brand'], brand_mapping = weighted_target_encoding(
    df,
    feature="brand",
    target="Price",
    m=20
)

In [32]:
df['model'].nunique()

143

In [33]:
model_mapping

{'1.2': 581186.6918953011,
 '3 Series': 624424.7871333964,
 '5 Series': 612710.5014191107,
 'A-Class Limousine': 849178.2059000601,
 'Alcazar': 700405.478627333,
 'Altroz': 580961.4474929044,
 'Amaze': 467398.5550872968,
 'Amaze 2nd Gen': 566121.6887417218,
 'Ameo': 571905.478627333,
 'Aspire': 581615.2633238726,
 'Aura': 587041.8422636966,
 'Aveo U-VA': 565329.5490381583,
 'BS IV': 571186.6918953011,
 'BSVI': 590567.6442762535,
 'Beat': 573472.4061810154,
 'Bolero Maxi Truck Plus': 591520.0252286345,
 'Bolero Neo': 603996.5447739706,
 'Brio': 449997.72942289495,
 'CLA': 694632.7513546057,
 'CR-V': 596541.8422636966,
 'Captur': 582520.0252286345,
 'Carens': 668041.8422636966,
 'Carnival': 654377.1680857773,
 'City': 647642.7208440902,
 'City Hybrid': 655520.0252286345,
 'Compass': 714578.0816064944,
 'Corolla Altis': 583723.6604455147,
 'Creta': 775435.844370861,
 'Curvv': 724205.0220750552,
 'Duster': 564170.4578174489,
 'EON': 405430.28458922496,
 'Ecosport': 564638.9879436237,
 'Ela

In [34]:
brand_mapping

{'Audi': 860996.387718242,
 'BMW': 947612.3280692816,
 'Chevrolet': 514080.02207505517,
 'Citroen': 599615.2633238726,
 'Datsun': 431872.5165562914,
 'Era': 579377.1680857773,
 'Ford': 627665.108427477,
 'HTX': 622948.596657206,
 'Honda': 559475.6498625941,
 'Hyundai': 501896.6266556291,
 'Jeep': 714578.0816064944,
 'Kia': 877780.8810826375,
 'LXI': 571186.6918953011,
 'MG': 768811.8714741231,
 'Mahindra': 672387.417218543,
 'Maruti': 431452.0160063651,
 'Mercedes-Benz': 1145876.821192053,
 'Nissan': 540330.6843267108,
 'Renault': 396434.20209356974,
 'S': 577472.4061810154,
 'Skoda': 731576.8560474033,
 'Sportz': 579405.478627333,
 'Tata': 690816.543555782,
 'Toyota': 954962.8666035951,
 'VX': 575450.9331727874,
 'VXI': 581186.6918953011,
 'Volkswagen': 678296.1814851345,
 'XZ': 599424.7871333964,
 'i': 584377.1680857773}

In [35]:
invalid_brands = [
    'Era', 'HTX', 'LXI', 'S', 'Sportz', 'VX', 'VXI', 'XZ', 'i'
]

df = df[~df['brand'].isin(invalid_brands)]

2. Registration Year

In [36]:
def clean_registration_year(x):
    if pd.isna(x):
        return None
    
    x = str(x)
    
    # extract 4-digit year
    match = re.search(r'\b(19|20)\d{2}\b', x)
    
    if match:
        return int(match.group())
    
    return None

In [37]:
df["Registration Year"] = df["Registration Year"].apply(clean_registration_year)

In [38]:
df['Registration Year'] = df['Registration Year'].fillna(df['Registration Year'].median())

3. 'Kms Driven'

In [39]:
df["Kms Driven"] = (
    df["Kms Driven"]
    .str.replace("Kms", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(int)
)

In [40]:
df['Kms Driven'] = df['Kms Driven'].fillna(df['Kms Driven'].median())

4. 'Ownership'

In [41]:
ownership_map = {
    "First Owner": 1,
    "Second Owner": 2,
    "Third Owner": 3,
    "Fourth Owner": 4,
    "Fifth Owner": 5
}

df["Ownership"] = df["Ownership"].map(ownership_map)
df['Ownership'] = df['Ownership'].fillna(df['Ownership'].median())

5. 'Fuel'

In [42]:
df['Fuel'] = df['Fuel'].fillna(df['Fuel'].mode()[0])

In [43]:
# get sorted fuel categories
fuel_categories = sorted(df['Fuel'].dropna().unique())

print("All Fuel Categories:", fuel_categories)

# first category will be dropped
dropped_category = fuel_categories[0]
print("Dropped Category:", dropped_category)

# apply one hot encoding
fuel_dummies = pd.get_dummies(df['Fuel'], prefix="Fuel")

# drop the first category column
fuel_dummies.drop(f"Fuel_{dropped_category}", axis=1, inplace=True)

# merge with dataframe
df = pd.concat([df, fuel_dummies], axis=1)

# remove original column
df.drop(columns=['Fuel'], inplace=True)

All Fuel Categories: ['CNG', 'Diesel', 'Petrol']
Dropped Category: CNG


6. 'Transmission'

In [44]:
df['Transmission'] = df['Transmission'].fillna(df['Transmission'].mode()[0])

transmission_map = {
    "Automatic": 1,
    "Manual": 0
}

df["Transmission"] = df["Transmission"].map(transmission_map)


7. 'Drive Type'

In [45]:
def clean_drive_type(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()

    if "fwd" in x or "front" in x:
        return "FWD"
    
    elif "rwd" in x:
        return "RWD"
    
    elif "awd" in x or "4wd" in x or "4x4" in x:
        return "AWD"
    
    elif "2wd" in x or "2 wd" in x or "two wheel" in x or "4x2" in x:
        return "FWD"
    
    else:
        return None


df["Drive Type"] = df["Drive Type"].apply(clean_drive_type)

In [46]:
df['Drive Type'] = df['Drive Type'].fillna(df['Drive Type'].mode()[0])

In [47]:
# get sorted fuel categories
Drive_Type__categories = sorted(df['Drive Type'].dropna().unique())

print("All Fuel Categories:", Drive_Type__categories)

# first category will be dropped
dropped_category = Drive_Type__categories[0]
print("Dropped Category:", dropped_category)

# apply one hot encoding
Drive_Type_dummies = pd.get_dummies(df['Drive Type'], prefix="Drive_Type")

# drop the first category column
Drive_Type_dummies.drop(f"Drive_Type_{dropped_category}", axis=1, inplace=True)

# merge with dataframe
df = pd.concat([df, Drive_Type_dummies], axis=1)

# remove original column
df.drop(columns=['Drive Type'], inplace=True)

All Fuel Categories: ['AWD', 'FWD', 'RWD']
Dropped Category: AWD


8. 'Engine 

In [48]:
df["Engine"] = (
    df["Engine"]
    .str.replace("cc", "", regex=False)
    .str.strip()
    .astype(float)
)
df['Engine'] = df['Engine'].fillna(df['Engine'].median())

9. Power

In [49]:
df["Power"] = (
    df["Power"]
    .str.replace("bhp", "", regex=False)
    .str.strip()
    .astype(float)
)
df['Power'] = df['Power'].fillna(df['Power'].median())

10. Mileage

In [50]:
df["Mileage"] = (
    df["Mileage"]
    .str.replace(r"[^\d.]", "", regex=True)
    .astype(float)
)

df["Mileage"] = df["Mileage"].fillna(df["Mileage"].median())

11. No. of Cylinders

In [51]:
df['No. of Cylinders'] =  df['No. of Cylinders'].fillna(df['No. of Cylinders'].median())


12. 'Turbo Charger'

In [52]:
df['Turbo Charger'] = df['Turbo Charger'].fillna(df['Turbo Charger'].mode()[0])

tubo_map = {
    "Yes": 1,
    "No": 0
}
df['Turbo Charger'] = df['Turbo Charger'].map(tubo_map)

13. Seats

In [53]:
df['Seats'].fillna(df['Seats'].mode()[0], inplace=True)

df["Seats"] = (
    df["Seats"]
    .str.replace("Seats", "", regex=False)
    .str.strip()
    .astype(float)
)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_20996\3059800891.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Seats'].fillna(df['Seats'].mode()[0], inplace=True)


14. Kerb Weight

In [54]:
df["Kerb Weight"] = (
    df["Kerb Weight"]
    .str.replace(r"[^\d]", "", regex=True)
)

# convert to numeric
df["Kerb Weight"] = pd.to_numeric(df["Kerb Weight"], errors="coerce")

# fill missing values
df["Kerb Weight"] = df["Kerb Weight"].fillna(df["Kerb Weight"].median())

15. Ground Clearance Unladen

In [55]:
df["Ground Clearance Unladen"] = (
    df["Ground Clearance Unladen"]
    .str.replace("mm", "", regex=False)
    .str.strip()
    .astype(float)
)

df["Ground Clearance Unladen"] = df["Ground Clearance Unladen"].fillna(
    df["Ground Clearance Unladen"].median()
)

In [56]:
cols = [
    "Petrol Fuel Tank Capacity",
    "Diesel Fuel Tank Capacity",
    "CNG Fuel Tank Capacity"
]

for col in cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[^\d.]", "", regex=True)  # keep numbers and decimal
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [57]:
petrol_median = df.loc[df["Fuel_Petrol"] == 1, "Petrol Fuel Tank Capacity"].median()

diesel_median = df.loc[df["Fuel_Diesel"] == 1, "Diesel Fuel Tank Capacity"].median()

cng_median = df.loc[
    (df["Fuel_Diesel"] == 0) & (df["Fuel_Petrol"] == 0),
    "CNG Fuel Tank Capacity"
].median()

In [58]:
df.loc[df["Fuel_Petrol"] == 1, "Petrol Fuel Tank Capacity"] = \
df.loc[df["Fuel_Petrol"] == 1, "Petrol Fuel Tank Capacity"].fillna(petrol_median)

In [59]:
df.loc[df["Fuel_Diesel"] == 1, "Diesel Fuel Tank Capacity"] = \
df.loc[df["Fuel_Diesel"] == 1, "Diesel Fuel Tank Capacity"].fillna(diesel_median)

In [60]:
df.loc[
    (df["Fuel_Diesel"] == 0) & (df["Fuel_Petrol"] == 0),
    "CNG Fuel Tank Capacity"
] = df.loc[
    (df["Fuel_Diesel"] == 0) & (df["Fuel_Petrol"] == 0),
    "CNG Fuel Tank Capacity"
].fillna(cng_median)

In [61]:
df[cols] = df[cols].fillna(0)

---

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score, mean_absolute_error

In [63]:
bool_features = ['Fuel_Diesel', 'Fuel_Petrol', 'Drive_Type_FWD', 'Drive_Type_RWD']

def bool_to_int_converter(df):
    return df.astype(int)

preprocessor = ColumnTransformer(
    transformers=[
        ('bool_to_int', FunctionTransformer(bool_to_int_converter), bool_features)
    ],
    remainder='passthrough'  # Keeps all other features in the dataset
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])

In [64]:
param_grid = {
    'regressor__n_estimators': [100, 500],
    'regressor__max_depth': [3, 6, 10],
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

In [65]:
X = df.drop(columns=['Price', 'car_name'], axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [66]:
grid_search.fit(X_train, y_train)

# Output results
print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Score: -63535373419.24703
Best Params: {'regressor__learning_rate': 0.01, 'regressor__max_depth': 10, 'regressor__n_estimators': 500, 'regressor__subsample': 0.8}


In [ ]:
# 1. Extract the best model from the grid search
best_model = grid_search.best_estimator_

# 2. Make predictions on the test set
y_pred = best_model.predict(X_test)

# 3. Quick evaluation
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):,.2f}")

R2 Score: 0.7953
Mean Absolute Error: 78,672.75
